In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import torch

In [ ]:
master_df = pd.read_csv("assests/mplads_master_analytics_ready.csv")
master_df.drop(columns=["Sr. No.", "Sr. No._sanctioned"], axis=1, inplace=True)

C:\Users\goure\AppData\Local\Temp\ipykernel_16356\2450069041.py:1: DtypeWarning: Columns (0: Sr. No., 1: Sr. No._sanctioned) have mixed types. Specify dtype option on import or set low_memory=False.
  master_df = pd.read_csv("assests/mplads_master_analytics_ready.csv")


In [3]:
# Automatically select GPU if available, otherwise fallback to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Initializing NLP Engine on Device: {device.upper()}")

nlp_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2", device=device
)


def run_mplads_ai_audit(
    master_df, similarity_threshold: float = 0.70
):
    df = master_df.copy()

    # =========================================================================
    # 1. ISOLATION FOREST: FINANCIAL ANOMALY ENGINE
    # =========================================================================
    print("\n[*] [Engine 1/2] Executing Isolation Forest Anomaly Detection...")

    df["Sanction Amount ( ₹ )"] = df["Sanction Amount ( ₹ )"].fillna(0.0)
    df["total_expenditure_released"] = df["total_expenditure_released"].fillna(
        0.0
    )
    df["cost_variance"] = df["cost_variance"].fillna(0.0)

    feature_cols = [
        "Sanction Amount ( ₹ )",
        "total_expenditure_released",
        "cost_variance",
    ]
    X = df[feature_cols]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    iso_forest = IsolationForest(
        contamination=0.15, random_state=42, n_jobs=-1
    )
    df["cost_anomaly_flag"] = iso_forest.fit_predict(X_scaled)

    raw_scores = iso_forest.decision_function(X_scaled)
    min_s, max_s = raw_scores.min(), raw_scores.max()
    df["cost_anomaly_risk_score"] = np.round(
        (1.0 - ((raw_scores - min_s) / (max_s - min_s + 1e-6))) * 100, 2
    )

    # =========================================================================
    # 2. NLP: GPU-ACCELERATED VECTORIZED DUPLICATE DETECTION
    # =========================================================================
    print(
        "[*] [Engine 2/2] Executing GPU-Accelerated NLP Similarity Search..."
    )

    df["Work Description"] = df["Work Description"].fillna("UNKNOWN WORK")
    df["Constituency"] = df["Constituency"].fillna("UNKNOWN DISTRICT")

    flagged_duplicates = []

    for district_name, group in df.groupby("Constituency"):
        num_records = len(group)
        if num_records < 2:
            continue

        work_names = group["Work Description"].tolist()
        work_ids = group["Work"].tolist()
        mp_ids = (
            group["Hon'Ble Members Of Parliament"].tolist()
            if "Hon'Ble Members Of Parliament" in group.columns
            else ["N/A"] * num_records
        )

        # Batch encode on GPU (batch_size 256 for fast parallel execution)
        embeddings = nlp_model.encode(
            work_names,
            convert_to_tensor=True,
            batch_size=256,
            show_progress_bar=False,
            device=device,
        )

        # Normalize embeddings on GPU tensor space
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

        # Fast GPU Matrix Multiplication
        sim_matrix = torch.mm(embeddings, embeddings.T)

        # Extract upper triangle matches above the similarity threshold
        upper_tri = torch.triu(sim_matrix, diagonal=1)
        match_indices = (
            (upper_tri >= similarity_threshold).nonzero(as_tuple=False).cpu()
        )

        # Append flagged pairs
        for idx in match_indices:
            i, j = idx[0].item(), idx[1].item()
            text_sim = float(sim_matrix[i, j].item())
            dup_risk = text_sim * 100.0

            flagged_duplicates.append({
                "original_work_id": work_ids[i],
                "original_work_name": work_names[i],
                "flagged_work_id": work_ids[j],
                "flagged_work_name": work_names[j],
                "mp_id": mp_ids[i],
                "district": district_name,
                "text_similarity_pct": round(text_sim * 100, 2),
                "duplicate_risk_score": round(dup_risk, 2),
            })

        # Clear PyTorch GPU cache per district to maintain low VRAM consumption
        del embeddings, sim_matrix, upper_tri
        if device == "cuda":
            torch.cuda.empty_cache()

    duplicates_df = pd.DataFrame(flagged_duplicates)

    # =========================================================================
    # 3. COMPOSITE AUDIT SCORE CALCULATOR
    # =========================================================================
    if not duplicates_df.empty:
        dup_summary = (
            duplicates_df.groupby("flagged_work_id")["duplicate_risk_score"]
            .max()
            .reset_index()
        )
        df = pd.merge(
            df,
            dup_summary,
            left_on="Work",
            right_on="flagged_work_id",
            how="left",
        )
        df["duplicate_risk_score"] = df["duplicate_risk_score"].fillna(0.0)
        df.drop(columns=["flagged_work_id"], inplace=True, errors="ignore")
    else:
        df["duplicate_risk_score"] = 0.0

    df["composite_risk_score"] = np.round(
        (df["cost_anomaly_risk_score"] * 0.5)
        + (df["duplicate_risk_score"] * 0.5),
        2,
    )

    print("[+] GPU Audit Engine Execution Complete!")
    return df, duplicates_df

[*] Initializing NLP Engine on Device: CUDA


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
run_mplads_ai_audit(master_df)


[*] [Engine 1/2] Executing Isolation Forest Anomaly Detection...
[*] [Engine 2/2] Executing GPU-Accelerated NLP Similarity Search...
[+] GPU Audit Engine Execution Complete!


(            Work Category                                               Work  \
 0           NORMAL/OTHERS  WS/\t MP620/2024-2025/133166-CONSTRUCTION OF B...   
 1       TRUST AND SOCIETY  WS/\t MP620/2025-2026/133167-CONSTRUCTION OF R...   
 2       TRUST AND SOCIETY  WS/\t MP620/2024-2025/133190-CONSTRUCTION OF B...   
 3           NORMAL/OTHERS  WS/\t MP620/2025-2026/133191-CONSTRUCTION OF B...   
 4           NORMAL/OTHERS  WS/\t MP620/2024-2025/133301-CONSTRUCTION OF B...   
 ...                   ...                                                ...   
 107085      NORMAL/OTHERS  NA-PROVIDING CCTV CAMERA SYSTEM FOR SECURITY O...   
 107086      NORMAL/OTHERS  NA-CONSTRUCTION OF COMMUNITY CENTERS AND COMMU...   
 107087      NORMAL/OTHERS  NA-CONSTRUCTION OF ROADS, LINK ROADS, PATHWAYS...   
 107088      NORMAL/OTHERS             NA-INSTALLING TUBE-WELLS AND BOREWELLS   
 107089                NaN                                                NaN   
 
             State        